## 1) Configuration

In [ ]:
# Cell 1: imports and configuration
from pathlib import Path
import pandas as pd
import numpy as np

PARENT_DIR = Path.cwd().parent
RESULTS_ROOT = PARENT_DIR / "Results"

MODELS = ["birdnet", "perch"]
AGGREGATIONS = ["onset", "gap"]

DATASET_CONFIG = {
    "chiffchaff": {
        "allowed_splits": {"withinyear", "acrossyear"},
    },
    "pipit": {
        "allowed_splits": {"withinyear", "acrossyear"},
    },
    "littleowl": {
        "allowed_splits": {"acrossyear"},
    },
    "littlepenguin": {
        "allowed_splits": {"withinyear"},
    },
    "kiwi": {
        "allowed_splits": {"acrossyear"},
    },
    "rtbc": {
        "allowed_splits": {"acrossyear"},
    },
    "greatTit": {
        "allowed_splits": {"acrossyear"},
    },
}

REQUIRED_COLS = [
    "run_name",
    "seed",
    "dataset_name",
    "split_type",
    "model",
    "aggregation",
    "embedding_dim",
    "test_accuracy",
    "test_f1_macro",
    "test_roc_auc_macro",
]

print("RESULTS_ROOT:", RESULTS_ROOT)

RESULTS_ROOT: /teamspace/studios/this_studio/Results


## 2) Find all files with metrics

In [2]:
# Cell 2: collect the expected metrics files
metric_files = []

for dataset_name, cfg in DATASET_CONFIG.items():
    for split_type in sorted(cfg["allowed_splits"]):
        metrics_dir = RESULTS_ROOT / dataset_name / split_type / "Metrics"

        for model in MODELS:
            for aggregation in AGGREGATIONS:
                metrics_file = metrics_dir / f"{dataset_name}_{split_type}_{model}_{aggregation}_final_metrics.csv"
                if metrics_file.exists():
                    metric_files.append(
                        {
                            "dataset_name": dataset_name,
                            "split_type": split_type,
                            "model": model,
                            "aggregation": aggregation,
                            "path": metrics_file,
                        }
                    )

metric_files_df = pd.DataFrame(metric_files)

print(f"Found {len(metric_files_df)} metrics files.")
display(metric_files_df)

Found 36 metrics files.


,dataset_name,split_type,model,aggregation,path
0,chiffchaff,acrossyear,birdnet,onset,/teamspace/studios/this_studio/Results/chiffch...
1,chiffchaff,acrossyear,birdnet,gap,/teamspace/studios/this_studio/Results/chiffch...
2,chiffchaff,acrossyear,perch,onset,/teamspace/studios/this_studio/Results/chiffch...
3,chiffchaff,acrossyear,perch,gap,/teamspace/studios/this_studio/Results/chiffch...
4,chiffchaff,withinyear,birdnet,onset,/teamspace/studios/this_studio/Results/chiffch...
5,chiffchaff,withinyear,birdnet,gap,/teamspace/studios/this_studio/Results/chiffch...
6,chiffchaff,withinyear,perch,onset,/teamspace/studios/this_studio/Results/chiffch...
7,chiffchaff,withinyear,perch,gap,/teamspace/studios/this_studio/Results/chiffch...
8,pipit,acrossyear,birdnet,onset,/teamspace/studios/this_studio/Results/pipit/a...
9,pipit,acrossyear,birdnet,gap,/teamspace/studios/this_studio/Results/pipit/a...


## 3) load everithing in a single Dataframe

In [3]:
# Cell 3: load and combine all metrics files
all_metrics = []

for row in metric_files:
    file_path = row["path"]

    try:
        df = pd.read_csv(file_path)

        # Normalize typo if needed
        if "aggegration" in df.columns and "aggregation" not in df.columns:
            df = df.rename(columns={"aggegration": "aggregation"})

        # Fill missing metadata from path config if needed
        if "dataset_name" not in df.columns:
            df["dataset_name"] = row["dataset_name"]
        if "split_type" not in df.columns:
            df["split_type"] = row["split_type"]
        if "model" not in df.columns:
            df["model"] = row["model"]
        if "aggregation" not in df.columns:
            df["aggregation"] = row["aggregation"]

        missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
        if missing_cols:
            print(f"Skipping {file_path.name}: missing columns {missing_cols}")
            continue

        # df["metrics_file"] = file_path.name
        # df["metrics_path"] = str(file_path)

        all_metrics.append(df)

    except Exception as e:
        print(f"Error reading {file_path}: {e}")

if not all_metrics:
    raise ValueError("No valid metrics files were loaded.")

combined_metrics_df = pd.concat(all_metrics, ignore_index=True)

combined_metrics_df = (
    combined_metrics_df
    .drop_duplicates(subset=["run_name", "seed"], keep="last")
    .sort_values(by=["dataset_name", "split_type", "model", "aggregation", "seed"])
    .reset_index(drop=True)
)

print(f"Combined rows: {len(combined_metrics_df)}")
display(combined_metrics_df.head())
display(combined_metrics_df.tail())

Combined rows: 181


,run_name,seed,dataset_name,split_type,model,aggregation,embedding_dim,hidden_dim,learning_rate,val_accuracy,...,test_accuracy,test_balanced_accuracy,test_recall_macro,test_recall_weighted,test_f1_macro,test_f1_weighted,test_roc_auc_macro,split_mode,test_cmc1,test_map5
0,chiffchaff_acrossyear_birdnet_gap,18.0,chiffchaff,acrossyear,birdnet,gap,1024.0,256.0,0.001,0.952381,...,0.942857,0.946159,0.946159,0.942857,0.947820,0.943039,0.999225,NaN,NaN,NaN
1,chiffchaff_acrossyear_birdnet_gap,23.0,chiffchaff,acrossyear,birdnet,gap,1024.0,256.0,0.001,0.990476,...,0.952381,0.947106,0.947106,0.952381,0.949965,0.952508,0.998552,NaN,NaN,NaN
2,chiffchaff_acrossyear_birdnet_gap,86.0,chiffchaff,acrossyear,birdnet,gap,1024.0,256.0,0.001,0.980952,...,0.980952,0.981818,0.981818,0.980952,0.980414,0.980456,0.999516,NaN,NaN,NaN
3,chiffchaff_acrossyear_birdnet_gap,123.0,chiffchaff,acrossyear,birdnet,gap,1024.0,256.0,0.001,0.980952,...,0.980952,0.985714,0.985714,0.980952,0.983974,0.981013,0.999082,NaN,NaN,NaN
4,chiffchaff_acrossyear_birdnet_gap,714.0,chiffchaff,acrossyear,birdnet,gap,1024.0,256.0,0.001,1.000000,...,0.971429,0.960714,0.960714,0.971429,0.966522,0.970249,0.999838,NaN,NaN,NaN


,run_name,seed,dataset_name,split_type,model,aggregation,embedding_dim,hidden_dim,learning_rate,val_accuracy,...,test_accuracy,test_balanced_accuracy,test_recall_macro,test_recall_weighted,test_f1_macro,test_f1_weighted,test_roc_auc_macro,split_mode,test_cmc1,test_map5
176,rtbc_acrossyear_perch_onset,13.0,rtbc,acrossyear,perch,onset,1280.0,256.0,0.001,0.977591,...,0.955182,0.913476,0.913476,0.955182,0.931622,0.954603,0.998718,NaN,NaN,NaN
177,rtbc_acrossyear_perch_onset,18.0,rtbc,acrossyear,perch,onset,1280.0,256.0,0.001,0.974790,...,0.966387,0.903573,0.903573,0.966387,0.928057,0.965571,0.999673,NaN,NaN,NaN
178,rtbc_acrossyear_perch_onset,46.0,rtbc,acrossyear,perch,onset,1280.0,256.0,0.001,0.969188,...,0.957983,0.947155,0.947155,0.957983,0.958428,0.957676,0.998894,NaN,NaN,NaN
179,rtbc_acrossyear_perch_onset,123.0,rtbc,acrossyear,perch,onset,1280.0,256.0,0.001,0.974790,...,0.969188,0.980130,0.980130,0.969188,0.983086,0.969148,0.999864,NaN,NaN,NaN
180,rtbc_acrossyear_perch_onset,321.0,rtbc,acrossyear,perch,onset,1280.0,256.0,0.001,0.974790,...,0.963585,0.947727,0.947727,0.963585,0.950758,0.963440,0.999230,NaN,NaN,NaN


## 4) Save unified dataframe

In [ ]:
# Cell 4: save combined dataframe
OUTPUT_DIR = PARENT_DIR / "Results" / "_combined_fcn"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_METRICS_FILE = OUTPUT_DIR / "all_metrics_onset_gap_combined.csv"
combined_metrics_df.to_csv(COMBINED_METRICS_FILE, index=False)

print(f"Saved combined metrics to: {COMBINED_METRICS_FILE}")

Saved combined metrics to: /teamspace/studios/this_studio/Results/_combined_fcn/all_metrics_onset_gap_combined.csv


## 5) numeric summary (mean and SD)

In [5]:
# Cell 5: numeric summary (mean and SD)
def sample_sd(x):
    return x.std(ddof=1) if len(x) > 1 else np.nan

summary_df = (
    combined_metrics_df
    .groupby(
        ["dataset_name", "split_type", "model", "aggregation", "embedding_dim"],
        as_index=False
    )
    .agg(
        n_seeds=("seed", "nunique"),
        test_accuracy_mean=("test_accuracy", "mean"),
        test_accuracy_sd=("test_accuracy", sample_sd),
        test_f1_macro_mean=("test_f1_macro", "mean"),
        test_f1_macro_sd=("test_f1_macro", sample_sd),
        test_roc_auc_macro_mean=("test_roc_auc_macro", "mean"),
        test_roc_auc_macro_sd=("test_roc_auc_macro", sample_sd),
    )
    .sort_values(by=["dataset_name", "split_type", "model", "aggregation"])
    .reset_index(drop=True)
)

display(summary_df)

,dataset_name,split_type,model,aggregation,embedding_dim,n_seeds,test_accuracy_mean,test_accuracy_sd,test_f1_macro_mean,test_f1_macro_sd,test_roc_auc_macro_mean,test_roc_auc_macro_sd
0,chiffchaff,acrossyear,birdnet,gap,1024.0,5,0.965714,0.017301,0.965739,0.016721,0.999243,0.000483
1,chiffchaff,acrossyear,birdnet,onset,1024.0,5,0.944762,0.027272,0.947048,0.024731,0.998770,0.000609
2,chiffchaff,acrossyear,perch,gap,1280.0,5,0.841905,0.043435,0.833170,0.048792,0.985573,0.010165
3,chiffchaff,acrossyear,perch,onset,1280.0,5,0.889524,0.029814,0.895332,0.033083,0.992598,0.003442
4,chiffchaff,withinyear,birdnet,gap,1024.0,5,0.974199,0.008748,0.966240,0.013923,0.999562,0.000235
5,chiffchaff,withinyear,birdnet,onset,1024.0,6,0.968616,0.002452,0.960173,0.003285,0.999310,0.000227
6,chiffchaff,withinyear,perch,gap,1280.0,5,0.938462,0.003323,0.922847,0.006346,0.997264,0.000479
7,chiffchaff,withinyear,perch,onset,1280.0,5,0.937500,0.010943,0.923075,0.014464,0.997141,0.000920
8,greatTit,acrossyear,birdnet,gap,1024.0,5,0.957140,0.005619,0.922622,0.010820,0.999588,0.000181
9,greatTit,acrossyear,birdnet,onset,1024.0,5,0.961111,0.005427,0.926477,0.009392,0.999654,0.000216


## 6) pretty summary (mean ± SD)

In [ ]:
# Cell 6: pretty summary (mean ± SD)
pretty_summary_df = summary_df.copy()

pretty_summary_df["test_accuracy"] = pretty_summary_df.apply(
    lambda r: f"{r['test_accuracy_mean']:.3f} ± {r['test_accuracy_sd']:.3f}"
    if pd.notna(r["test_accuracy_sd"]) else f"{r['test_accuracy_mean']:.3f} ± NA",
    axis=1
)

pretty_summary_df["test_f1_macro"] = pretty_summary_df.apply(
    lambda r: f"{r['test_f1_macro_mean']:.3f} ± {r['test_f1_macro_sd']:.3f}"
    if pd.notna(r["test_f1_macro_sd"]) else f"{r['test_f1_macro_mean']:.3f} ± NA",
    axis=1
)

pretty_summary_df["test_roc_auc_macro"] = pretty_summary_df.apply(
    lambda r: f"{r['test_roc_auc_macro_mean']:.4f} ± {r['test_roc_auc_macro_sd']:.4f}"
    if pd.notna(r["test_roc_auc_macro_sd"]) else f"{r['test_roc_auc_macro_mean']:.4f} ± NA",
    axis=1
)

pretty_summary_df = pretty_summary_df[
    [
        "dataset_name",
        "split_type",
        "model",
        "aggregation",
        "embedding_dim",
        "n_seeds",
        "test_accuracy",
        "test_f1_macro",
        "test_roc_auc_macro",
    ]
]

display(pretty_summary_df)

,dataset_name,split_type,model,aggregation,embedding_dim,n_seeds,test_accuracy,test_f1_macro,test_roc_auc_macro
0,chiffchaff,acrossyear,birdnet,gap,1024.0,5,0.966 ± 0.017,0.966 ± 0.017,0.9992 ± 0.0005
1,chiffchaff,acrossyear,birdnet,onset,1024.0,5,0.945 ± 0.027,0.947 ± 0.025,0.9988 ± 0.0006
2,chiffchaff,acrossyear,perch,gap,1280.0,5,0.842 ± 0.043,0.833 ± 0.049,0.9856 ± 0.0102
3,chiffchaff,acrossyear,perch,onset,1280.0,5,0.890 ± 0.030,0.895 ± 0.033,0.9926 ± 0.0034
4,chiffchaff,withinyear,birdnet,gap,1024.0,5,0.974 ± 0.009,0.966 ± 0.014,0.9996 ± 0.0002
5,chiffchaff,withinyear,birdnet,onset,1024.0,6,0.969 ± 0.002,0.960 ± 0.003,0.9993 ± 0.0002
6,chiffchaff,withinyear,perch,gap,1280.0,5,0.938 ± 0.003,0.923 ± 0.006,0.9973 ± 0.0005
7,chiffchaff,withinyear,perch,onset,1280.0,5,0.938 ± 0.011,0.923 ± 0.014,0.9971 ± 0.0009
8,greatTit,acrossyear,birdnet,gap,1024.0,5,0.957 ± 0.006,0.923 ± 0.011,0.9996 ± 0.0002
9,greatTit,acrossyear,birdnet,onset,1024.0,5,0.961 ± 0.005,0.926 ± 0.009,0.9997 ± 0.0002


## 7) save summaries

In [7]:
SUMMARY_FILE = OUTPUT_DIR / "all_metrics_onset_gap_summary_numeric.csv"
PRETTY_SUMMARY_FILE = OUTPUT_DIR / "all_metrics_onset_gap_summary_pretty.csv"

summary_df.to_csv(SUMMARY_FILE, index=False)
pretty_summary_df.to_csv(PRETTY_SUMMARY_FILE, index=False)

print(f"Saved numeric summary to: {SUMMARY_FILE}")
print(f"Saved pretty summary to: {PRETTY_SUMMARY_FILE}")

Saved numeric summary to: /teamspace/studios/this_studio/Results/_combined_fcn/all_metrics_onset_gap_summary_numeric.csv
Saved pretty summary to: /teamspace/studios/this_studio/Results/_combined_fcn/all_metrics_onset_gap_summary_pretty.csv


## 8) pivot table for quick comparison

### f1 macro

In [8]:
# Optional Cell 8: pivot table for quick comparison
pivot_df = pretty_summary_df.pivot_table(
    index=["dataset_name", "split_type"],
    columns=["model", "aggregation"],
    values="test_f1_macro",
    aggfunc="first"
)

display(pivot_df)

model                           birdnet                         perch  \
aggregation                         gap          onset            gap   
dataset_name  split_type                                                
chiffchaff    acrossyear  0.966 ± 0.017  0.947 ± 0.025  0.833 ± 0.049   
              withinyear  0.966 ± 0.014  0.960 ± 0.003  0.923 ± 0.006   
greatTit      acrossyear  0.923 ± 0.011  0.926 ± 0.009  0.863 ± 0.009   
kiwi          acrossyear  0.923 ± 0.008  0.790 ± 0.036  0.917 ± 0.039   
littleowl     acrossyear  0.979 ± 0.010  0.979 ± 0.010  0.975 ± 0.003   
littlepenguin withinyear  0.931 ± 0.011  0.934 ± 0.009  0.922 ± 0.010   
pipit         acrossyear  0.955 ± 0.019  0.958 ± 0.020  0.899 ± 0.024   
              withinyear  0.950 ± 0.031  0.940 ± 0.037  0.904 ± 0.034   
rtbc          acrossyear  0.947 ± 0.037  0.947 ± 0.037  0.950 ± 0.022   

model                                    
aggregation                       onset  
dataset_name  split_type                 
chiffchaff    acrossyear  0.895 ± 0.033  
              withinyear  0.923 ± 0.014  
greatTit      acrossyear  0.859 ± 0.008  
kiwi          acrossyear  0.848 ± 0.048  
littleowl     acrossyear  0.975 ± 0.003  
littlepenguin withinyear  0.922 ± 0.010  
pipit         acrossyear  0.905 ± 0.023  
              withinyear  0.898 ± 0.036  
rtbc          acrossyear  0.950 ± 0.022

### accuracy

In [9]:
pivot_df = pretty_summary_df.pivot_table(
    index=["dataset_name", "split_type"],
    columns=["model", "aggregation"],
    values="test_accuracy",
    aggfunc="first"
)

display(pivot_df)

model                           birdnet                         perch  \
aggregation                         gap          onset            gap   
dataset_name  split_type                                                
chiffchaff    acrossyear  0.966 ± 0.017  0.945 ± 0.027  0.842 ± 0.043   
              withinyear  0.974 ± 0.009  0.969 ± 0.002  0.938 ± 0.003   
greatTit      acrossyear  0.957 ± 0.006  0.961 ± 0.005  0.915 ± 0.005   
kiwi          acrossyear  0.923 ± 0.008  0.837 ± 0.029  0.925 ± 0.030   
littleowl     acrossyear  0.979 ± 0.010  0.979 ± 0.010  0.974 ± 0.004   
littlepenguin withinyear  0.939 ± 0.007  0.942 ± 0.016  0.926 ± 0.008   
pipit         acrossyear  0.957 ± 0.016  0.957 ± 0.020  0.908 ± 0.021   
              withinyear  0.954 ± 0.027  0.945 ± 0.030  0.915 ± 0.031   
rtbc          acrossyear  0.964 ± 0.009  0.964 ± 0.009  0.962 ± 0.006   

model                                    
aggregation                       onset  
dataset_name  split_type                 
chiffchaff    acrossyear  0.890 ± 0.030  
              withinyear  0.938 ± 0.011  
greatTit      acrossyear  0.915 ± 0.006  
kiwi          acrossyear  0.866 ± 0.024  
littleowl     acrossyear  0.974 ± 0.004  
littlepenguin withinyear  0.926 ± 0.008  
pipit         acrossyear  0.910 ± 0.020  
              withinyear  0.909 ± 0.031  
rtbc          acrossyear  0.962 ± 0.006

### ROC_AUC_macro

In [10]:
pivot_df = pretty_summary_df.pivot_table(
    index=["dataset_name", "split_type"],
    columns=["model", "aggregation"],
    values="test_roc_auc_macro",
    aggfunc="first"
)

display(pivot_df)

model                             birdnet                             perch  \
aggregation                           gap            onset              gap   
dataset_name  split_type                                                      
chiffchaff    acrossyear  0.9992 ± 0.0005  0.9988 ± 0.0006  0.9856 ± 0.0102   
              withinyear  0.9996 ± 0.0002  0.9993 ± 0.0002  0.9973 ± 0.0005   
greatTit      acrossyear  0.9996 ± 0.0002  0.9997 ± 0.0002  0.9988 ± 0.0003   
kiwi          acrossyear  0.9987 ± 0.0008  0.9893 ± 0.0038  0.9989 ± 0.0009   
littleowl     acrossyear  0.9997 ± 0.0003  0.9997 ± 0.0003  0.9995 ± 0.0003   
littlepenguin withinyear  0.9984 ± 0.0005  0.9983 ± 0.0008  0.9971 ± 0.0008   
pipit         acrossyear  0.9979 ± 0.0012  0.9986 ± 0.0011  0.9915 ± 0.0035   
              withinyear  0.9982 ± 0.0015  0.9986 ± 0.0013  0.9924 ± 0.0036   
rtbc          acrossyear  0.9991 ± 0.0007  0.9991 ± 0.0007  0.9993 ± 0.0005   

model                                      
aggregation                         onset  
dataset_name  split_type                   
chiffchaff    acrossyear  0.9926 ± 0.0034  
              withinyear  0.9971 ± 0.0009  
greatTit      acrossyear  0.9989 ± 0.0003  
kiwi          acrossyear  0.9925 ± 0.0034  
littleowl     acrossyear  0.9995 ± 0.0003  
littlepenguin withinyear  0.9971 ± 0.0008  
pipit         acrossyear  0.9926 ± 0.0060  
              withinyear  0.9939 ± 0.0045  
rtbc          acrossyear  0.9993 ± 0.0005